# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from pydantic import BaseModel,Field
from dotenv import load_dotenv
from typing import List, Dict, Annotated

In [3]:
# TODO: Load environment variables

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
# 1. Definir el nombre exacto que usarás después
class retrievedDocument(BaseModel):
    """Pydantic data model for retrieved documents from the vectorstore"""
    Platform: Annotated[str, Field(description="Game platform (PC, Xbox, Wii...)")]
    Name: Annotated[str, Field(description="Title of the game")]
    YearOfRelease: Annotated[int, Field(description="Year of release")]
    Description: Annotated[str, Field(description="One-liner game description")]

@tool
def retrieve_game(query:str, num_matches:int=5) -> List[retrievedDocument]:
    """
    Semantic search: Finds most results in the vector DB
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")
    collection = chroma_client.get_collection("udaplay")
    vectorstore_db = VectorStore(chroma_collection=collection)
    
    out = vectorstore_db.query(
        query_texts=[query],   
        n_results=num_matches
    )

    documents = []
    if out and out['metadatas']:
        # Usamos el nombre correcto de la clase: retrievedDocument
        metadatas = out['metadatas'][0]
        documents = [retrievedDocument(**metadata) for metadata in metadatas]
        
    return documents

@tool
def retrieve_all_games() -> List[retrievedDocument]:
    """
    Retrieves all records from the database
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")
    collection = chroma_client.get_collection("udaplay")
    all_records = collection.get()

    documents = []
    if all_records and all_records['metadatas']:
        metadatas = all_records['metadatas']
        documents = [retrievedDocument(**metadata) for metadata in metadatas]
        
    return documents


#### Evaluate Retrieval Tool

In [7]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class evaluationResult(BaseModel):
    """pydantic data model for evaluation result"""
    useful: Annotated[bool, Field(description="whether the documents are useful to answer the question")]
    description: Annotated[str, Field(description="description about the evaluation result")]

@tool
def evaluate_retrieval(question:str, retrieved_docs:List[retrievedDocument])->evaluationResult:
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
    args: 
    - question: original question from user
    - retrieved_docs: Full context of documents available to answer the question
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """

    llm = LLM(
        api_key=OPENAI_API_KEY, model='gpt-4o', temperature=0  )
    
    system_prompt = ("You are a data validation expert." 
    "Your mission is to determine if the retrieved documents contain the necessary information to answer the user's question." 
    "Critically evaluate whether the content is sufficient, relevant, and accurate before confirming its utility."
    )
    
    user_prompt= ("Evaluate this output: "
    f"user question: {question}\n"
    f"retrieved documents:\n{retrieved_docs}"
    )

    messages=[
        SystemMessage(content=system_prompt),
        UserMessage(content=user_prompt)
    ]

    response = llm.invoke(
        input=messages, 
        response_format=evaluationResult
        )

    return response.content   

#### Game Web Search Tool

In [9]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API
    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)
    
    # Perform the search
    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    result = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }
    
    return result

### Agent

In [12]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

system_prompt = (
    "You are a computer games expert. Your goal is to provide faithful, structured information following this strict protocol:\n"
    "1. **Search**: Query the VectorDB (collections of records).\n"
    "2. **Reason & Refine**: Analyze results; if insufficient, rephrase and retry the semantic search.\n"
    "3. **Fallback**: Use web search only if the internal DB fails.\n"
    "4. **Audit**: You MUST run the `evaluate_retrieval` tool on your gathered data before your final output.\n"
    "5. **Output**: Provide a concise bulleted list with citations. If info is missing after all steps, state 'I don't know'. Avoid external assumptions."
)


agent = Agent(
    model_name="gpt-4o",
    tools=tools,
    instructions=system_prompt
)

In [14]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

query = "When Pokémon Gold and Silver was released?"
response = agent.invoke(
    query=query,
    session_id='question_0'
    )

query = "Which one was the first 3D platformer Mario game?"
response = agent.invoke(
    query=query,
    session_id='question_2'
    )

query = "Was Mortal Kombat X realeased for Playstation 5?"
response = agent.invoke(
    query=query,
    session_id='question_3'
    )

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: voc-1204**************************************2299. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes